In [ ]:
#ruff: noqa

In [14]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
%autoreload 2

In [16]:
import os
import sys

module_path = os.path.abspath(os.path.join('..', ''))
if module_path not in sys.path:
	sys.path.append(module_path)

from rect_prune_ood.lib.util.paths import get_top_level_path

In [17]:
import os
import pickle
import sys
import tarfile

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from rect_prune_ood.lib.models.resnet_cifar_react import resnet50_cifar

In [22]:
# Define a custom dataset for CIFAR-10 that reads from the .tar.gz file
class CIFAR10CustomDataset(Dataset):
	def __init__(self, tar_path, batch_files, transform=None):
		self.tar_path = tar_path
		self.batch_files = batch_files
		self.transform = transform
		self.data = []
		self.labels = []

		# Load data from tar.gz file
		self._load_data()

	def _load_data(self):
		with tarfile.open(self.tar_path, 'r:gz') as tar:
			for batch_file in self.batch_files:
				file = tar.extractfile(batch_file)
				if file is not None:
					batch = pickle.load(file, encoding='latin1')  # Load batch
					self.data.append(batch['data'])
					self.labels.extend(batch['labels'])

		# Convert to numpy arrays
		self.data = np.vstack(self.data).reshape(-1, 3, 32, 32).astype("float32")
		self.labels = np.array(self.labels)

	def __len__(self):
		return len(self.labels)

	def __getitem__(self, idx):
		image = self.data[idx]
		label = self.labels[idx]

		if self.transform:
			image = self.transform(torch.tensor(image))  # Apply transforms
		return image, label


data_dir = os.path.join(
	get_top_level_path(), "datasets", "id_datasets", "cifar10", "cifar-10-python.tar.gz"
)  # Adjust path as needed
print(data_dir)
train_files = [
	"cifar-10-batches-py" + os.sep + f"data_batch_{i}" for i in range(1, 6)
]  # Batches 1-5 for training
test_file = "cifar-10-batches-py" + os.sep + "test_batch"  # Test batch

/workspaces/RectPruneOOD/datasets/id_datasets/cifar10/cifar-10-python.tar.gz


In [23]:
# Define transformations
transform = transforms.Compose(
	[
		transforms.ToPILImage(),  # Convert numpy array to PIL image
		transforms.RandomHorizontalFlip(),
		transforms.RandomCrop(32, padding=4),
		transforms.ToTensor(),
		transforms.Normalize(
			(0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)
		),  # CIFAR-10 mean/std
	]
)

# test and train
train_dataset = CIFAR10CustomDataset(data_dir, train_files, transform=transform)
test_dataset = CIFAR10CustomDataset(data_dir, [test_file], transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

In [24]:
# config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = resnet50_cifar(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


num_epochs = 10
best_accuracy = 0.0

In [ ]:
from tqdm import tqdm

print("Starting Training...")
for epoch in tqdm(range(num_epochs)):
	model.train()
	running_loss = 0.0
	print("Iterating train loader")
	for inputs, labels in tqdm(train_loader):
		inputs, labels = inputs.to(device), labels.to(device)

		optimizer.zero_grad()
		outputs = model(inputs)
		loss = criterion(outputs, labels)
		loss.backward()
		optimizer.step()

		running_loss += loss.item()

	print(
		f"Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}"
	)

	# Val
	model.eval()
	correct = 0
	total = 0
	with torch.no_grad():
		for inputs, labels in test_loader:
			inputs, labels = inputs.to(device), labels.to(device)
			outputs = model(inputs)
			_, predicted = torch.max(outputs, 1)
			total += labels.size(0)
			correct += (predicted == labels).sum().item()

	accuracy = 100 * correct / total
	print(f"Validation Accuracy: {accuracy:.2f}%")

	# Save weights
	if accuracy > best_accuracy:
		best_accuracy = accuracy
		torch.save(model.state_dict(), 'best_resnet50_cifar10.pth')
		print("Model weights saved.")

print("Training complete.")

Starting Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Iterating train loader


100%|██████████| 391/391 [00:46<00:00,  8.50it/s]

Epoch [1/10], Loss: 1.5641


Validation Accuracy: 49.70%


... (3 outputs left out)

Epoch [2/10], Loss: 1.0441


Validation Accuracy: 63.55%


 20%|██        | 2/10 [01:38<06:33, 49.13s/it]

Model weights saved.
Iterating train loader
